# ICBHI: End-to-end pretrained ResNet18 binary classifier

This notebook fine-tunes an ImageNet-pretrained ResNet18 to classify each annotated respiratory-cycle segment as **Normal (0)** or **Abnormal (1)**.

## Evaluation protocol
- The `split` column in `split_metadata.csv` must identify the existing official ICBHI `train` and `test` assignments.
- The official test rows are kept untouched. Validation patients are selected **only from official training rows**, using patient-level grouping.
- Patient ID is derived from the ICBHI filename convention: the portion of `source_file` before the first underscore (e.g. `101` from `101_1b1_Al_sc_Meditron`).
- The checkpoint with the highest validation HS-score is selected. The last epoch is saved separately.
- The official test set is evaluated only after training, for both checkpoints.

**Important:** Verify that `split_metadata.csv` was created from the official ICBHI patient-level split. The notebook checks that a patient does not occur in both official train and test. If your metadata uses different split names or filename conventions, fix the path/columns and patient-ID extraction rather than bypassing the checks.

In [1]:
# 1. Imports and reproducibility
import os
import random
import sys
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [2]:
# 2. Paths and training configuration
DATA_DIR = Path("../datasets/audios")
METADATA_PATH = Path("../datasets/split_metadata.csv")
OUTPUT_DIR = Path("models/resnet18_end_to_end")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
N_MELS = 32
N_FFT = 1024
HOP_LENGTH = 256
WIN_LENGTH = 1024
FMIN = 50
FMAX = 2_000
IMAGE_SIZE = 224

VAL_SIZE = 0.20             # fraction of official training patients
BATCH_SIZE = 16
NUM_WORKERS = 0
EPOCHS = 20
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
USE_AMP = (device.type == "cuda")

BEST_CHECKPOINT = OUTPUT_DIR / "resnet18_best_val_hs.pt"
LAST_CHECKPOINT = OUTPUT_DIR / "resnet18_last_epoch.pt"
RESULTS_PATH = OUTPUT_DIR / "test_results.json"

print("Audio directory:", DATA_DIR.resolve())
print("Metadata:", METADATA_PATH.resolve())
print("Output directory:", OUTPUT_DIR.resolve())

Audio directory: D:\.JKU CS courses\Master Thesis\datasets\raw\ICBHI_final_database\audios
Metadata: D:\.JKU CS courses\Master Thesis\datasets\split_metadata.csv
Output directory: D:\.JKU CS courses\Master Thesis\src\models\resnet18_icbhi_binary


In [3]:
# 3. Load and validate metadata
if not METADATA_PATH.exists():
    raise FileNotFoundError(
        f"Metadata file not found: {METADATA_PATH.resolve()}. "
        "Set METADATA_PATH to your split_metadata.csv."
    )
if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Audio directory not found: {DATA_DIR.resolve()}. Set DATA_DIR to the folder "
        "containing the ICBHI .wav files."
    )

df = pd.read_csv(METADATA_PATH)
required_columns = {"source_file", "start", "end", "crackle", "wheeze", "split"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"split_metadata.csv is missing required columns: {sorted(missing_columns)}")

df = df.copy()
df["source_file"] = df["source_file"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip().str.lower()

split_aliases = {"train": "train", "training": "train", "test": "test", "testing": "test"}
unknown_splits = sorted(set(df["split"]) - set(split_aliases))
if unknown_splits:
    raise ValueError(
        f"Unexpected split values: {unknown_splits}. Expected train/test labels. "
        "Map them explicitly only after verifying the official split."
    )
df["split"] = df["split"].map(split_aliases)

# ICBHI filenames conventionally start with the patient ID followed by an underscore.
def patient_id_from_source(source_file):
    stem = Path(source_file).stem
    if "_" not in stem:
        raise ValueError(
            f"Cannot infer patient ID from filename '{source_file}'. Expected an ICBHI "
            "filename beginning with patient ID followed by '_'. Define patient IDs "
            "explicitly in metadata if your naming convention differs."
        )
    return stem.split("_", 1)[0]

df["patient_id"] = df["source_file"].map(patient_id_from_source)

if df[["source_file", "start", "end", "crackle", "wheeze", "split"]].isna().any().any():
    raise ValueError("Metadata contains missing values in required columns.")

official_train = df[df["split"] == "train"].copy()
official_test = df[df["split"] == "test"].copy()
if official_train.empty or official_test.empty:
    raise ValueError("Both official train and official test rows must be present in metadata.")

train_patients = set(official_train["patient_id"])
test_patients = set(official_test["patient_id"])
overlap = train_patients & test_patients
""" 
since there is data overlap in the official data split, the error << raise ValueError() >>
will be ignored and replaced with a warning message << warnings.warn() >>
"""
if overlap:
    warnings.warn(
        f"\n!!! WARNING DATA LEAKAGE !!!\nPatient leakage in metadata: {len(overlap)} patient(s) appear in "
        f"both official train and test, e.g. {sorted(overlap)[:10]}. Correct split_metadata.csv before training!"
    )

print(f"Rows: {len(df):,}")
print(f"Official train rows/patients: {len(official_train):,} / {official_train['patient_id'].nunique():,}")
print(f"Official test rows/patients:  {len(official_test):,} / {official_test['patient_id'].nunique():,}")
print("Official split patient overlap:", len(overlap))
display(df.groupby("split").agg(rows=("source_file", "size"), patients=("patient_id", "nunique")))

Rows: 6,898
Official train rows/patients: 4,142 / 79
Official test rows/patients:  2,756 / 49
Official split patient overlap: 2


C:\Users\arebr\AppData\Local\Temp\ipykernel_36340\3746613056.py:61: UserWarning: 
!!! WARNING DATA LEAKAGE !!!
Patient leakage in metadata: 2 patient(s) appear in both official train and test, e.g. ['156', '218']. Correct split_metadata.csv before training!
  warnings.warn(


,rows,patients
split,,
test,2756,49
train,4142,79


In [4]:
# 4. Create a patient-level validation split from official training patients only
gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
train_idx, val_idx = next(
    gss.split(official_train, groups=official_train["patient_id"])
)
train_df = official_train.iloc[train_idx].reset_index(drop=True)
val_df = official_train.iloc[val_idx].reset_index(drop=True)
test_df = official_test.reset_index(drop=True)  # held out until final evaluation

train_patient_set = set(train_df["patient_id"])
val_patient_set = set(val_df["patient_id"])
test_patient_set = set(test_df["patient_id"])

assert train_patient_set.isdisjoint(val_patient_set), "Train/validation patient leakage."

print(f"Train: {len(train_df):,} segments, {train_df['patient_id'].nunique()} patients")
print(f"Val:   {len(val_df):,} segments, {val_df['patient_id'].nunique()} patients")
print(f"Test:  {len(test_df):,} segments, {test_df['patient_id'].nunique()} patients (official test)")
display(pd.DataFrame([
    {"split": "train", "segments": len(train_df), "patients": train_df["patient_id"].nunique()},
    {"split": "validation", "segments": len(val_df), "patients": val_df["patient_id"].nunique()},
    {"split": "official_test", "segments": len(test_df), "patients": test_df["patient_id"].nunique()},
]))

Train: 3,515 segments, 63 patients
Val:   627 segments, 16 patients
Test:  2,756 segments, 49 patients (official test)


,split,segments,patients
0,train,3515,63
1,validation,627,16
2,official_test,2756,49


In [5]:
# 5. Binary labels: Normal=0, Abnormal=1
# ICBHI annotation convention: crackle/wheeze are 0 or 1.
def binary_label(crackle, wheeze):
    return 0 if int(crackle) == 0 and int(wheeze) == 0 else 1

for frame in (train_df, val_df, test_df):
    frame["label"] = [
        binary_label(c, w) for c, w in zip(frame["crackle"], frame["wheeze"])
    ]

print("Training label counts:")
display(train_df["label"].map({0: "Normal", 1: "Abnormal"}).value_counts().rename_axis("class").to_frame("segments"))
print("Validation label counts:")
display(val_df["label"].map({0: "Normal", 1: "Abnormal"}).value_counts().rename_axis("class").to_frame("segments"))

Training label counts:


,segments
class,
Abnormal,1916
Normal,1599


Validation label counts:


,segments
class,
Normal,464
Abnormal,163


In [6]:
# 6. Dataset: load audio, crop the annotated segment, compute normalized log-mel image
class ICBHISpectrogramDataset(Dataset):
    def __init__(self, frame, audio_dir, train=False):
        self.frame = frame.reset_index(drop=True).copy()
        self.audio_dir = Path(audio_dir)
        self.train = train
        self.waveform_cache = {}

    def __len__(self):
        return len(self.frame)

    def _load_audio(self, source_file):
        filename = source_file if source_file.lower().endswith(".wav") else f"{source_file}.wav"
        wav_path = self.audio_dir / filename
        if not wav_path.exists():
            raise FileNotFoundError(f"Audio file referenced in metadata not found: {wav_path}")
        if filename not in self.waveform_cache:
            audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            self.waveform_cache[filename] = audio
        return self.waveform_cache[filename]

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        audio = self._load_audio(row["source_file"])
        start = max(0, int(float(row["start"]) * SAMPLE_RATE))
        end = min(len(audio), int(float(row["end"]) * SAMPLE_RATE))
        segment = audio[start:end]
        if segment.size < 1000:
            # This should not happen after metadata validation; fail rather than silently alter labels.
            raise ValueError(
                f"Segment too short ({segment.size} samples): {row['source_file']} "
                f"[{row['start']}, {row['end']}]"
            )

        mel = librosa.feature.melspectrogram(
            y=segment, sr=SAMPLE_RATE, n_mels=N_MELS,
            n_fft=N_FFT, hop_length=HOP_LENGTH, win_length=WIN_LENGTH,
            fmin=FMIN, fmax=FMAX, power=2.0
        )
        mel_db = librosa.power_to_db(mel, ref=np.max)
        mel_db = (mel_db - mel_db.mean()) / (mel_db.std() + 1e-6)

        # [1, mel, time] then resize; repeat to 3 channels for pretrained ResNet18.
        spec = torch.from_numpy(mel_db.astype(np.float32)).unsqueeze(0).unsqueeze(0)
        spec = torch.nn.functional.interpolate(
            spec, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False
        ).squeeze(0)
        image = spec.repeat(3, 1, 1)

        # Apply ImageNet normalization expected by the pretrained weights.
        mean = torch.tensor([0.485, 0.456, 0.406], dtype=image.dtype).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225], dtype=image.dtype).view(3, 1, 1)
        image = (image - mean) / std

        label = torch.tensor(int(row["label"]), dtype=torch.long)
        return image, label

train_dataset = ICBHISpectrogramDataset(train_df, DATA_DIR, train=True)
val_dataset = ICBHISpectrogramDataset(val_df, DATA_DIR, train=False)
test_dataset = ICBHISpectrogramDataset(test_df, DATA_DIR, train=False)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda")
)

# Quick shape / label sanity check before starting training.
images, labels = next(iter(train_loader))
print("Batch images:", tuple(images.shape), "labels:", tuple(labels.shape))
print("Batch labels:", labels.tolist())

Batch images: (16, 3, 224, 224) labels: (16,)
Batch labels: [1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1]


In [7]:
# 7. Metrics (abnormal is the positive class)
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    hs_score = (
        2 * sensitivity * specificity / (sensitivity + specificity)
        if (sensitivity + specificity) else 0.0
    )
    accuracy = accuracy_score(y_true, y_pred)
    return {
        "loss": None,
        "accuracy": float(accuracy),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "HS_score": float(hs_score),
        "confusion_matrix": cm.tolist(),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }

def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    running_loss = 0.0
    all_targets, all_preds = [], []

    for images, targets in tqdm(loader, leave=False):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(
                device_type=device.type, dtype=torch.float16,
                enabled=(USE_AMP and device.type == "cuda")
            ):
                logits = model(images)
                loss = criterion(logits, targets)

            if training:
                if scaler is not None and USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = logits.argmax(dim=1)
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_preds.extend(preds.detach().cpu().numpy().tolist())

    metrics = compute_metrics(all_targets, all_preds)
    metrics["loss"] = float(running_loss / len(loader.dataset))
    return metrics

In [8]:
# 8. Initialize pretrained ResNet18 and optimizer
weights = ResNet18_Weights.DEFAULT
model = resnet18(weights=weights)
model.fc = nn.Linear(model.fc.in_features, 2)  # Normal / Abnormal
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable parameters: 11,177,538


In [10]:
# 9. Train; select best checkpoint using validation HS-score
history = []
best_val_hs = -1.0
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer, scaler=scaler)
    val_metrics = run_epoch(model, val_loader, criterion, optimizer=None)

    scheduler.step(val_metrics["HS_score"])
    current_lr = optimizer.param_groups[0]["lr"]

    row = {
        "epoch": epoch,
        "lr": current_lr,
        **{f"train_{k}": v for k, v in train_metrics.items() if k != "confusion_matrix"},
        **{f"val_{k}": v for k, v in val_metrics.items() if k != "confusion_matrix"},
    }
    history.append(row)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_metrics": val_metrics,
        "seed": SEED,
        "class_mapping": {"Normal": 0, "Abnormal": 1},
        "config": {
            "sample_rate": SAMPLE_RATE, "n_mels": N_MELS, "n_fft": N_FFT,
            "hop_length": HOP_LENGTH, "image_size": IMAGE_SIZE,
            "learning_rate": LEARNING_RATE, "batch_size": BATCH_SIZE,
        },
    }

    # Save the last epoch every epoch (overwritten each time).
    torch.save(checkpoint, LAST_CHECKPOINT)

    # Save best validation HS-score checkpoint.
    if val_metrics["HS_score"] > best_val_hs:
        best_val_hs = val_metrics["HS_score"]
        best_epoch = epoch
        torch.save(checkpoint, BEST_CHECKPOINT)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
        f"HS={train_metrics['HS_score']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
        f"SE={val_metrics['sensitivity']:.4f}, SP={val_metrics['specificity']:.4f}, "
        f"HS={val_metrics['HS_score']:.4f} | lr={current_lr:.2e}"
    )

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Best epoch:", best_epoch, "| best validation HS-score:", round(best_val_hs, 4))
print("Best checkpoint:", BEST_CHECKPOINT.resolve())
print("Last checkpoint:", LAST_CHECKPOINT.resolve())

Epoch 01/20 | train loss=0.6056, acc=0.6825, HS=0.6723 | val loss=0.7346, acc=0.5582, SE=0.6503, SP=0.5259, HS=0.5815 | lr=1.00e-04


Epoch 02/20 | train loss=0.4290, acc=0.7986, HS=0.7951 | val loss=0.9149, acc=0.5694, SE=0.7485, SP=0.5065, HS=0.6041 | lr=1.00e-04


Epoch 03/20 | train loss=0.2518, acc=0.8962, HS=0.8958 | val loss=1.0152, acc=0.6156, SE=0.6319, SP=0.6099, HS=0.6207 | lr=1.00e-04


Epoch 04/20 | train loss=0.1717, acc=0.9354, HS=0.9356 | val loss=1.5338, acc=0.5582, SE=0.8098, SP=0.4698, HS=0.5947 | lr=1.00e-04


Epoch 07/20 | train loss=0.0891, acc=0.9670, HS=0.9670 | val loss=1.5813, acc=0.6427, SE=0.5521, SP=0.6746, HS=0.6072 | lr=1.00e-04


...

Epoch 17/20 | train loss=0.0063, acc=0.9986, HS=0.9985 | val loss=1.5782, acc=0.6332, SE=0.6135, SP=0.6401, HS=0.6265 | lr=5.00e-05


Epoch 19/20 | train loss=0.0053, acc=0.9983, HS=0.9983 | val loss=1.6218, acc=0.6555, SE=0.6748, SP=0.6487, HS=0.6615 | lr=2.50e-05


Epoch 20/20 | train loss=0.0074, acc=0.9980, HS=0.9979 | val loss=1.3880, acc=0.6571, SE=0.5828, SP=0.6832, HS=0.6290 | lr=2.50e-05


,epoch,lr,train_loss,train_accuracy,train_sensitivity,train_specificity,train_HS_score,train_tn,train_fp,train_fn,train_tp,val_loss,val_accuracy,val_sensitivity,val_specificity,val_HS_score,val_tn,val_fp,val_fn,val_tp
0,1,0.000100,0.605568,0.682504,0.735386,0.619137,0.672273,990,609,507,1409,0.734639,0.558214,0.650307,0.525862,0.581501,244,220,57,106
1,2,0.000100,0.429024,0.798578,0.823591,0.768605,0.795149,1229,370,338,1578,0.914887,0.569378,0.748466,0.506466,0.604132,235,229,41,122
2,3,0.000100,0.251847,0.896159,0.899791,0.891807,0.895782,1426,173,192,1724,1.015158,0.615630,0.631902,0.609914,0.620713,283,181,60,103
3,4,0.000100,0.171689,0.935420,0.933194,0.938086,0.935634,1500,99,128,1788,1.533802,0.558214,0.809816,0.469828,0.594656,218,246,31,132
4,5,0.000100,0.127598,0.951920,0.954071,0.949343,0.951701,1518,81,88,1828,1.139928,0.653907,0.638037,0.659483,0.648583,306,158,59,104
5,6,0.000100,0.105826,0.958464,0.956159,0.961226,0.958686,1537,62,84,1832,1.381739,0.615630,0.680982,0.592672,0.633766,275,189,52,111
6,7,0.000100,0.089088,0.966999,0.967119,0.966854,0.966987,1546,53,63,1853,1.581252,0.642743,0.552147,0.674569,0.607249,313,151,73,90
7,8,0.000100,0.087487,0.969275,0.969207,0.969356,0.969281,1550,49,59,1857,1.572632,0.641148,0.680982,0.627155,0.652961,291,173,52,111
8,9,0.000100,0.086946,0.968421,0.965553,0.971857,0.968695,1554,45,66,1850,1.735165,0.566188,0.730061,0.508621,0.599547,236,228,44,119
9,10,0.000100,0.071185,0.972119,0.974426,0.969356,0.971884,1550,49,49,1867,1.597554,0.598086,0.631902,0.586207,0.608197,272,192,60,103


Best epoch: 14 | best validation HS-score: 0.6712
Best checkpoint: D:\.JKU CS courses\Master Thesis\src\models\resnet18_icbhi_binary\resnet18_best_val_hs.pt
Last checkpoint: D:\.JKU CS courses\Master Thesis\src\models\resnet18_icbhi_binary\resnet18_last_epoch.pt


In [11]:
# 10. Final evaluation on the untouched official test set
# Do not use these results to change hyperparameters or select the model.
def evaluate_checkpoint(checkpoint_path, loader, name):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    eval_model = resnet18(weights=None)
    eval_model.fc = nn.Linear(eval_model.fc.in_features, 2)
    eval_model.load_state_dict(checkpoint["model_state_dict"])
    eval_model = eval_model.to(device)

    metrics = run_epoch(eval_model, loader, criterion, optimizer=None)
    print(f"\n{name} checkpoint (epoch {checkpoint['epoch']})")
    print(f"Test loss:       {metrics['loss']:.4f}")
    print(f"Test accuracy:   {metrics['accuracy']:.4f}")
    print(f"Sensitivity:     {metrics['sensitivity']:.4f}")
    print(f"Specificity:     {metrics['specificity']:.4f}")
    print(f"HS-score:        {metrics['HS_score']:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(metrics["confusion_matrix"]))
    return {
        "checkpoint": str(checkpoint_path),
        "epoch": int(checkpoint["epoch"]),
        "test_loss": metrics["loss"],
        "test_accuracy": metrics["accuracy"],
        "test_sensitivity": metrics["sensitivity"],
        "test_specificity": metrics["specificity"],
        "test_HS_score": metrics["HS_score"],
        "confusion_matrix": metrics["confusion_matrix"],
    }

best_test_results = evaluate_checkpoint(BEST_CHECKPOINT, test_loader, "BEST")
last_test_results = evaluate_checkpoint(LAST_CHECKPOINT, test_loader, "LAST")


BEST checkpoint (epoch 14)
Test loss:       1.7560
Test accuracy:   0.5809
Sensitivity:     0.4979
Specificity:     0.6428
HS-score:        0.5611
Confusion matrix [[TN, FP], [FN, TP]]:
[[1015  564]
 [ 591  586]]



LAST checkpoint (epoch 20)
Test loss:       1.8607
Test accuracy:   0.5976
Sensitivity:     0.4911
Specificity:     0.6770
HS-score:        0.5692
Confusion matrix [[TN, FP], [FN, TP]]:
[[1069  510]
 [ 599  578]]


In [12]:
# 11. Save final metrics and print a compact comparison
results = {
    "selection_metric": "validation HS-score",
    "best_epoch": int(best_epoch),
    "best_validation_HS_score": float(best_val_hs),
    "best_checkpoint_test": best_test_results,
    "last_checkpoint_test": last_test_results,
    "split_counts": {
        "train_segments": int(len(train_df)),
        "validation_segments": int(len(val_df)),
        "official_test_segments": int(len(test_df)),
        "train_patients": int(train_df["patient_id"].nunique()),
        "validation_patients": int(val_df["patient_id"].nunique()),
        "official_test_patients": int(test_df["patient_id"].nunique()),
    },
}
with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

comparison = pd.DataFrame([
    {
        "checkpoint": "best_validation_HS",
        "epoch": best_test_results["epoch"],
        "test_loss": best_test_results["test_loss"],
        "accuracy": best_test_results["test_accuracy"],
        "sensitivity": best_test_results["test_sensitivity"],
        "specificity": best_test_results["test_specificity"],
        "HS_score": best_test_results["test_HS_score"],
    },
    {
        "checkpoint": "last_epoch",
        "epoch": last_test_results["epoch"],
        "test_loss": last_test_results["test_loss"],
        "accuracy": last_test_results["test_accuracy"],
        "sensitivity": last_test_results["test_sensitivity"],
        "specificity": last_test_results["test_specificity"],
        "HS_score": last_test_results["test_HS_score"],
    },
])
display(comparison)
print("Saved final test metrics to:", RESULTS_PATH.resolve())

,checkpoint,epoch,test_loss,accuracy,sensitivity,specificity,HS_score
0,best_validation_HS,14,1.756003,0.580914,0.497876,0.642812,0.561136
1,last_epoch,20,1.860693,0.597605,0.491079,0.677011,0.569247


Saved final test metrics to: D:\.JKU CS courses\Master Thesis\src\models\resnet18_icbhi_binary\test_results.json
